# Zuber: análisis de viajes en taxi en Chicago

**Autor:** Juan Pablo Quintero

## Descripción del proyecto

Zuber es una nueva empresa de viajes compartidos que se lanza en Chicago. Como analista, mi tarea es encontrar patrones en los datos disponibles para entender las preferencias de los pasajeros y el impacto de factores externos en los viajes.

Para eso voy a:

1. Obtener por web scraping los datos del clima de Chicago en noviembre de 2017.
2. Analizar con SQL los datos de las empresas de taxis de la competencia.
3. Probar la hipótesis de que el clima afecta la duración de los viajes desde el Loop hasta el Aeropuerto Internacional O'Hare.


## Descripción de los datos

La base de datos contiene información sobre viajes en taxi en Chicago:

| Tabla             | Contenido            | Campos principales                                                                                                           |
| ----------------- | -------------------- | ---------------------------------------------------------------------------------------------------------------------------- |
| `neighborhoods`   | Barrios de la ciudad | `neighborhood_id`, `name`                                                                                                    |
| `cabs`            | Taxis                | `cab_id`, `vehicle_id`, `company_name`                                                                                       |
| `trips`           | Viajes               | `trip_id`, `cab_id`, `start_ts`, `end_ts`, `duration_seconds`, `distance_miles`, `pickup_location_id`, `dropoff_location_id` |
| `weather_records` | Clima por hora       | `record_id`, `ts`, `temperature`, `description`                                                                              |

**Nota:** `trips` y `weather_records` no tienen una relación directa. Se unen por la hora de inicio del viaje (`trips.start_ts`) y la hora del registro meteorológico (`weather_records.ts`).


## Importar librerías


In [1]:
# Web scraping
import requests
from bs4 import BeautifulSoup

# Manipulación de datos
import pandas as pd

---

## Paso 1. Web scraping del clima

**Objetivo:** extraer de una página web la tabla con el clima por hora de Chicago en noviembre de 2017.

**Para qué sirve:** estos registros son la base para clasificar cada hora como _Good_ o _Bad_ según el clima (Paso 3) y probar la hipótesis sobre la duración de los viajes (Paso 5).


In [2]:
# URL de la página con los registros del clima de Chicago (noviembre de 2017)
URL = 'https://practicum-content.s3.us-west-1.amazonaws.com/data-analyst-eng/moved_chicago_weather_2017.html'

# Petición GET: descarga la página. La respuesta queda en `req`
req = requests.get(URL)

# Parseo: convierte el HTML (req.text) en un objeto navegable usando el parser lxml
soup = BeautifulSoup(req.text, 'lxml')

In [3]:
# Ubicar la tabla por su id (find devuelve un solo elemento, no una lista)
table = soup.find('table', attrs={'id': 'weather_records'})

# Encabezados: texto de cada etiqueta <th>
heading_table = [row.text for row in table.find_all('th')]

# Filas de datos: recorre cada <tr> y salta la fila de encabezados (la que tiene <th>).
# De las demás toma el texto de cada celda <td>
content = []
for row in table.find_all('tr'):
    if not row.find_all('th'):
        content.append([element.text for element in row.find_all('td')])

In [4]:
# Construir el DataFrame con las filas extraídas y los encabezados como nombres de columna
weather_records = pd.DataFrame(content, columns=heading_table)

# Imprimir el DataFrame completo
print(weather_records)

           Date and time Temperature       Description
0    2017-11-01 00:00:00     276.150     broken clouds
1    2017-11-01 01:00:00     275.700  scattered clouds
2    2017-11-01 02:00:00     275.610   overcast clouds
3    2017-11-01 03:00:00     275.350     broken clouds
4    2017-11-01 04:00:00     275.240     broken clouds
..                   ...         ...               ...
692  2017-11-29 20:00:00     281.340        few clouds
693  2017-11-29 21:00:00     281.690      sky is clear
694  2017-11-29 22:00:00     281.070        few clouds
695  2017-11-29 23:00:00     280.060      sky is clear
696  2017-11-30 00:00:00     278.460      sky is clear

[697 rows x 3 columns]


### Revisión del DataFrame obtenido


In [5]:
# Dimensiones: (filas, columnas)
print(weather_records.shape)

(697, 3)


In [6]:
# Tipos de datos y valores no nulos por columna
weather_records.info()

<class 'pandas.DataFrame'>
RangeIndex: 697 entries, 0 to 696
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   Date and time  697 non-null    str  
 1   Temperature    697 non-null    str  
 2   Description    697 non-null    str  
dtypes: str(3)
memory usage: 42.0 KB


In [7]:
# Primeras y últimas filas, para confirmar que el rango de fechas es completo
display(weather_records.head())
display(weather_records.tail())

,Date and time,Temperature,Description
0,2017-11-01 00:00:00,276.150,broken clouds
1,2017-11-01 01:00:00,275.700,scattered clouds
2,2017-11-01 02:00:00,275.610,overcast clouds
3,2017-11-01 03:00:00,275.350,broken clouds
4,2017-11-01 04:00:00,275.240,broken clouds


,Date and time,Temperature,Description
692,2017-11-29 20:00:00,281.340,few clouds
693,2017-11-29 21:00:00,281.690,sky is clear
694,2017-11-29 22:00:00,281.070,few clouds
695,2017-11-29 23:00:00,280.060,sky is clear
696,2017-11-30 00:00:00,278.460,sky is clear


In [8]:
# Se guarda una copia en local de la tabla de la página del proyecto
weather_records.to_csv('../data/chicago_weather_nov_2017.csv', index=False)

### Observaciones del Paso 1

- **Volumen y período:** se obtuvieron **697 registros horarios**, desde el 1 de noviembre de 2017 a las 00:00 hasta el 30 de noviembre a las 00:00.

- **Calidad:** no hay valores nulos en ninguna de las 3 columnas ni filas vacías.

- **Tipos de datos:** el scraping devuelve todo como texto, y las 3 columnas quedaron como `str`. Para usarlas en un análisis habría que convertirlas:
  - `Date and time` → `datetime`
  - `Temperature` → numérico (`float`)
  - `Description` es categórica, por lo que puede quedarse como texto.

- **Unidad de temperatura:** por sible lógica los valores de temperatura no corresponden a °C ni a °F. Se consideran que están en **Kelvin**.

- **Relevancia para la hipótesis:** la columna `Description` es la que permite clasificar cada hora como _Bad_ (si contiene "rain" o "storm") o _Good_ para el Paso 3.


---

## Paso 2. Análisis exploratorio de datos (SQL)

Las consultas de este paso se ejecutaron en la base de datos PostgreSQL de la plataforma de TripleTen. Cada consulta también está guardada como archivo en la carpeta `sql/`.

**Técnicas usadas:** `INNER JOIN` entre `trips` y `cabs` por `cab_id`, filtros de fecha con `CAST(... AS DATE)`, agregación con `COUNT` y `GROUP BY`, filtrado de texto con `LIKE` y agrupación condicional con `CASE`.


### Ejercicio 1. Viajes por empresa el 15 y 16 de noviembre de 2017

**Pregunta:** ¿cuántos viajes hizo cada empresa de taxis entre el 15 y el 16 de noviembre de 2017?

```sql
-- Paso 2 · Ejercicio 1
-- Cantidad de viajes en taxi por empresa entre el 15 y el 16 de noviembre de 2017, ordenada de mayor a menor.

SELECT
    c.company_name,
    COUNT(t.trip_id) AS trips_amount
FROM
    trips t
    INNER JOIN cabs c ON t.cab_id = c.cab_id
WHERE
    CAST(t.start_ts AS DATE) BETWEEN '2017-11-15' AND '2017-11-16'
GROUP BY
    c.company_name
ORDER BY
    trips_amount DESC;
```

**Nota:** `start_ts` es un _timestamp_. Comparar directamente con `BETWEEN '2017-11-15' AND '2017-11-16'` dejaría fuera casi todo el día 16 (se interpretaría como `2017-11-16 00:00:00`). Por eso se convierte primero a fecha con `CAST(... AS DATE)`.

**Resultado (primeras 5 de 64 empresas):**

| company_name                    | trips_amount |
| ------------------------------- | -----------: |
| Flash Cab                       |       19,558 |
| Taxi Affiliation Services       |       11,422 |
| Medallion Leasin                |       10,367 |
| Yellow Cab                      |        9,888 |
| Taxi Affiliation Service Yellow |        9,299 |

**Observaciones:**

- **Mercado concentrado:** Flash Cab tiene casi el doble de viajes que la segunda empresa.

- **Calidad de los nombres:** Se observa que en el resultado obtenido hay nombres que podrían ser variantes de nombres de las empresas. Aún no se unifican porque no hay forma de confirmar que sean la misma empresa.


### Ejercicio 2. Empresas "Yellow" y "Blue" del 1 al 7 de noviembre de 2017

**Pregunta:** ¿cuántos viajes hizo cada empresa cuyo nombre contiene "Yellow" o "Blue" entre el 1 y el 7 de noviembre de 2017?

```sql
-- Paso 2 · Ejercicio 2
-- Cantidad de viajes por empresa cuyo nombre contiene "Yellow" o "Blue", entre el 1 y el 7 de noviembre de 2017.

SELECT
    c.company_name AS company_name,
    COUNT(t.trip_id) AS trips_amount
FROM
    trips t
    INNER JOIN cabs c ON t.cab_id = c.cab_id
WHERE
    CAST(t.start_ts AS DATE) BETWEEN '2017-11-01' AND '2017-11-07'
    -- Los paréntesis son necesarios: AND se evalúa antes que OR
    AND (c.company_name LIKE '%Yellow%' OR c.company_name LIKE '%Blue%')
GROUP BY
    c.company_name
ORDER BY
    trips_amount DESC;
```

**Resultado:**

| company_name                      | trips_amount |
| --------------------------------- | -----------: |
| Yellow Cab                        |       33,668 |
| Taxi Affiliation Service Yellow   |       29,213 |
| Blue Ribbon Taxi Association Inc. |       17,675 |
| Blue Diamond                      |        6,764 |

**Observaciones:**

- Las dos empresas "Yellow" están casi que parejas y ambas superan en gran cantidad de viajes a las "Blue".

- Los líderes del mercado como se observa en el anterior ejercicio, "Flash Cab" no tiene "nombre de color", por lo tanto, este filtro no captura a los competidores más fuertes.


### Ejercicio 3. Flash Cab, Taxi Affiliation Services y "Other" del 1 al 7 de noviembre de 2017

**Pregunta:** del 1 al 7 de noviembre de 2017, las empresas más populares fueron Flash Cab y Taxi Affiliation Services. ¿Cuántos viajes hizo cada una, frente a todas las demás empresas agrupadas como "Other"?

```sql
-- Paso 2 · Ejercicio 3
-- Viajes del 1 al 7 de noviembre de 2017 de las dos empresas más populares
-- (Flash Cab y Taxi Affiliation Services); el resto se agrupa como "Other".

SELECT
    CASE
        WHEN c.company_name IN ('Flash Cab', 'Taxi Affiliation Services') THEN c.company_name
        ELSE 'Other'
    END AS company,
    COUNT(t.trip_id) AS trips_amount
FROM
    trips t
    INNER JOIN cabs c ON c.cab_id = t.cab_id
WHERE
    CAST(t.start_ts AS DATE) BETWEEN '2017-11-01' AND '2017-11-07'
GROUP BY
    company
ORDER BY
    trips_amount DESC;
```

**Nota:** los nombres se comparan de forma exacta con `IN`, no con `LIKE`, además, se agrupa por la nueva columna `company`.

**Resultado:**

| company                   | trips_amount | % del total |
| ------------------------- | -----------: | ----------: |
| Other (62 empresas)       |      335,771 |      76.8 % |
| Flash Cab                 |       64,084 |      14.6 % |
| Taxi Affiliation Services |       37,583 |       8.6 % |
| **Total**                 |  **437,438** |   **100 %** |

**Observaciones:**

- Entre las dos empresas líderes suman casi una cuarta parte de todos los viajes de la semana (23.2 %). Las otras empresas se reparten el resto.

- Flash Cab hace casi el doble de los viajes de Taxi Affiliation Services. Para Zuber es el principal competidor como referencia.


---

## Paso 3. Consultas para probar la hipótesis (SQL)

**Hipótesis a preparar:** la duración de los viajes desde el Loop hasta el Aeropuerto Internacional O'Hare cambia los sábados lluviosos.

En este paso se construye, con tres consultas encadenadas, el dataset que se usa en la prueba de hipótesis (Paso 5): primero se identifican los barrios, luego se clasifica el clima de cada hora y al final se unen ambos con los viajes.

**Técnicas usadas:** escape de comillas en texto, `CASE` con `LIKE`, subconsulta dentro de un `JOIN`, unión de tablas sin llave directa (por hora) y extracción del día de la semana con `EXTRACT(ISODOW ...)`.


### Ejercicio 4. Identificadores de los barrios Loop y O'Hare

**Pregunta:** ¿cuáles son los `neighborhood_id` del Loop y de O'Hare?

```sql
-- Paso 3 · Ejercicio 4
-- Identificadores de los barrios Loop y O'Hare en la tabla neighborhoods.
-- La comilla simple de O'Hare se escapa duplicándola ('O''Hare').

SELECT
    neighborhood_id,
    name
FROM
    neighborhoods
WHERE
    name = 'O''Hare'
    OR name = 'Loop';
```

**Nota:** en SQL los textos van entre comillas simples. Como _O'Hare_ tiene una comilla simple dentro, se escapa duplicándola (`'O''Hare'`).

**Resultado:**

| neighborhood_id | name   |
| --------------: | ------ |
|              50 | Loop   |
|              63 | O'Hare |


### Ejercicio 5. Clasificación del clima por hora (Good / Bad)

**Pregunta:** para cada hora de `weather_records`, ¿el clima fue _Bad_ (la descripción contiene "rain" o "storm") o _Good_?

```sql
-- Paso 3 · Ejercicio 5
-- Clasificación de cada hora de weather_records según el clima:
-- "Bad" si la descripción contiene "rain" o "storm"; "Good" en cualquier otro caso.

SELECT
    ts,
    CASE
        WHEN description LIKE '%rain%' OR description LIKE '%storm%' THEN 'Bad'
        ELSE 'Good'
    END AS weather_conditions
FROM
    weather_records;
```

**Resultado (fragmento):** una fila por hora.

| ts                  | weather_conditions |
| ------------------- | ------------------ |
| 2017-11-02 02:00:00 | Good               |
| 2017-11-02 03:00:00 | Bad                |
| 2017-11-02 04:00:00 | Bad                |
| …                   | …                  |

**Observaciones:**

- Predominan ampliamente las horas _Good_.


### Ejercicio 6. Viajes del Loop a O'Hare los sábados, con su clima

**Pregunta:** ¿qué viajes empezaron un sábado en el Loop (50) y terminaron en O'Hare (63), con qué condición climática y cuánto duraron? Se ignoran los viajes sin datos de clima.

```sql
-- Paso 3 · Ejercicio 6
-- Viajes del Loop (50) a O'Hare (63) que empezaron un sábado, con la condición
-- climática de la hora de inicio y su duración (resultado exportado como project_sql_result_07.csv).
-- La subconsulta reutiliza la clasificación Good/Bad del ejercicio 5. El INNER JOIN
-- descarta los viajes sin registro de clima a esa hora.

SELECT
    t.start_ts,
    ws.weather_conditions,
    t.duration_seconds
FROM
    trips t
    INNER JOIN (
        SELECT
            ts,
            CASE
                WHEN description LIKE '%rain%' OR description LIKE '%storm%' THEN 'Bad'
                ELSE 'Good'
            END AS weather_conditions
        FROM
            weather_records
    ) AS ws ON ws.ts = t.start_ts
WHERE
    t.pickup_location_id = 50
    AND t.dropoff_location_id = 63
    -- ISODOW: lunes = 1 ... domingo = 7, por lo que el sábado es 6
    AND EXTRACT(ISODOW FROM t.start_ts) = 6
ORDER BY
    t.trip_id;
```

**Notas:**

- **Subconsulta:** la consulta del ejercicio 5 se usa como una tabla virtual (`ws`) con las columnas `ts` y `weather_conditions`. En PostgreSQL, una subconsulta dentro de un `JOIN` necesita alias obligatoriamente.

- **Unión sin llave directa:** `trips` y `weather_records` se unen por la hora (`t.start_ts = ws.ts`). Las dos están redondeadas a la hora, así que coinciden y no es necesario castear.

- **`INNER JOIN`:** cumple la condición de ignorar los viajes sin registro de clima.

- **Orden:** se ordena por `trip_id` aunque esa columna no aparezca en el `SELECT`.

- Solo aparecen los cuatro sábados de noviembre de 2017: **4, 11, 18 y 25**.

**Resultado (primeras filas):**

| start_ts            | weather_conditions | duration_seconds |
| ------------------- | ------------------ | ---------------: |
| 2017-11-25 12:00:00 | Good               |            1,380 |
| 2017-11-25 16:00:00 | Good               |            2,410 |
| 2017-11-25 14:00:00 | Good               |            1,920 |
| 2017-11-25 12:00:00 | Good               |            1,543 |
| 2017-11-04 10:00:00 | Good               |            2,512 |


---

## Paso 4. Análisis exploratorio de datos (Python)

> _Pendiente_


---

## Paso 5. Prueba de hipótesis

> _Pendiente_


---

## Conclusión general

> _Pendiente_
